# Movie Consumption Prediction Challenge
## Model development reference

This notebook documents how the provided baseline model was trained using movie metadata (dataset A) and consumption records (dataset C). The model predicts **June 2026 streams for each film × country × platform** from May 2026 inputs and movie attributes.

**Your assignment is not to retrain the model.** Read the transformations below and reproduce the raw-data preparation on the separate inference inputs. The provided `artifacts/movie_consumption_model.pkl` contains the trained estimator and fitted preprocessing, but **does not** perform the consumption aggregation or metadata join.

Dataset B is not used in this model because its availability snapshot is dated June. `TITLE_ID` identifies the film for joins and output; it is not a model feature.

### 1 · Inputs and environment

Read the notebook from the package root: file paths below are relative to that directory. The training-source files retain their original column names. Required library versions are listed in `requirements.txt`. Only load pickle files from a trusted source.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA = Path("data")
SEED = 42
MAY = "2026-05-01"
JUNE = "2026-06-01"

In [2]:
movies_raw = pd.read_csv(DATA / "train_movies.csv", encoding="utf-8-sig", dtype={"TITLE_ID": "string"})
consumption_raw = pd.read_csv(DATA / "train_consumption.csv", encoding="utf-8-sig", dtype={"imdb_id": "string"})

display(movies_raw[["TITLE_ID", "ORIGINAL_TITLE", "YEAR", "RUNTIME_MINUTES", "PRIMARY_GENRE", "RATING_VALUE", "RATING_VOTE_COUNT"]].head(3))
display(consumption_raw.head(3))

,TITLE_ID,ORIGINAL_TITLE,YEAR,RUNTIME_MINUTES,PRIMARY_GENRE,RATING_VALUE,RATING_VOTE_COUNT
0,tt0327785,The Killer's Game,2024,104.0,Action,6.0,21870.0
1,tt0439572,The Flash,2023,144.0,Action,7.0,252005.0
2,tt0498396,The Twits,2025,98.0,Animation,5.0,4623.0


,imdb_id,month,country,platform,streams,total_minutes
0,tt0327785,2026-05-01,Argentina,Amazon,27.0,2919.0
1,tt0327785,2026-05-01,Brazil,Amazon,7.0,0.0
2,tt0327785,2026-06-01,Brazil,Amazon,65.0,6113.0


### 2 · Standardize keys and data types

The movie dataset calls its key `TITLE_ID`; the consumption dataset calls the same key `imdb_id`. Align these names and parse the month and consumption metrics before working across datasets.

In [3]:
movies = movies_raw.rename(columns={
    "YEAR": "release_year",
    "RUNTIME_MINUTES": "runtime_minutes",
    "PRIMARY_GENRE": "primary_genre",
    "RATING_VALUE": "rating_value",
    "RATING_VOTE_COUNT": "rating_vote_count",
}).copy()

consumption = consumption_raw.rename(columns={"imdb_id": "TITLE_ID"}).copy()
consumption["month"] = pd.to_datetime(consumption["month"], errors="raise").dt.strftime("%Y-%m-%d")
for column in ("streams", "total_minutes"):
    consumption[column] = pd.to_numeric(consumption[column], errors="raise")

GRAIN = ["TITLE_ID", "country", "platform"]

### 3 · Aggregate May features at film × country × platform

Country and platform are **model features**, not counts of markets. Sum May streams and minutes **within each film–country–platform combination**; do not collapse the observations to a single row per film.

In [4]:
may_features = (
    consumption.loc[consumption["month"].eq(MAY)]
    .groupby(GRAIN, as_index=False, dropna=False)
    .agg(
        may_streams=("streams", "sum"),
        may_total_minutes=("total_minutes", "sum"),
    )
)
display(may_features.head(5))

,TITLE_ID,country,platform,may_streams,may_total_minutes
0,tt0327785,Argentina,Amazon,27.0,2919.0
1,tt0327785,Brazil,Amazon,7.0,0.0
2,tt0327785,Colombia,Amazon,13.0,1387.0
3,tt0439572,Brazil,Amazon,11.0,1634.0
4,tt0439572,Brazil,HBO Max,4.0,131.0


### 4 · Construct the target for model development

To train the model, match May and June consumption on **all three keys**: film, country, and platform. A film may have valid pairs in some markets but not others. Missing June records are not assumed to represent zero streams.

**Inference is different:** June outcomes are unavailable at prediction time. Generate predictions for **all observed May combinations** in the inference input; do not use June data to select those rows.

In [5]:
june_target = (
    consumption.loc[consumption["month"].eq(JUNE)]
    .groupby(GRAIN, as_index=False, dropna=False)
    .agg(june_streams=("streams", "sum"))
)
training_pairs = may_features.merge(june_target, on=GRAIN, how="inner", validate="one_to_one")
display(training_pairs.head(5))

,TITLE_ID,country,platform,may_streams,may_total_minutes,june_streams
0,tt0327785,Brazil,Amazon,7.0,0.0,65.0
1,tt0327785,Colombia,Amazon,13.0,1387.0,6.0
2,tt0439572,Brazil,Amazon,11.0,1634.0,29.0
3,tt0439572,Brazil,HBO Max,4.0,131.0,45.0
4,tt0439572,Colombia,HBO Max,13.0,1927.0,9.0


### 5 · Join movie attributes and select features

Add film-level attributes without changing the film–country–platform grain. Ratings and vote counts are included as popularity signals.

**Timing assumption:** dataset A has no capture timestamps for ratings and votes. For this exercise, assume these values were available at the end of May. A real forecasting system would need an as-of-May metadata snapshot to rule out look-ahead leakage.

In [6]:
film_attributes = movies[[
    "TITLE_ID", "release_year", "runtime_minutes", "primary_genre", "rating_value", "rating_vote_count"
]].copy()
training_table = training_pairs.merge(film_attributes, on="TITLE_ID", how="left", validate="many_to_one")

CATEGORICAL_FEATURES = ["country", "platform", "primary_genre"]
NUMERIC_FEATURES = [
    "may_streams", "may_total_minutes", "release_year", "runtime_minutes", "rating_value", "rating_vote_count"
]
FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES
TARGET = "june_streams"

display(training_table[["TITLE_ID"] + FEATURE_COLUMNS + [TARGET]].head(5))

,TITLE_ID,country,platform,primary_genre,may_streams,may_total_minutes,release_year,runtime_minutes,rating_value,rating_vote_count,june_streams
0,tt0327785,Brazil,Amazon,Action,7.0,0.0,2024,104.0,6.0,21870.0,65.0
1,tt0327785,Colombia,Amazon,Action,13.0,1387.0,2024,104.0,6.0,21870.0,6.0
2,tt0439572,Brazil,Amazon,Action,11.0,1634.0,2023,144.0,7.0,252005.0,29.0
3,tt0439572,Brazil,HBO Max,Action,4.0,131.0,2023,144.0,7.0,252005.0,45.0
4,tt0439572,Colombia,HBO Max,Action,13.0,1927.0,2023,144.0,7.0,252005.0,9.0


### 6 · Evaluate on unseen films (85/15 split)

Split by **film ID**, not by consumption row: every country/platform observation for a title remains on the same side. This avoids leaking film-level information between training and evaluation.

In [7]:
film_ids = np.array(sorted(training_table.TITLE_ID.unique().tolist()))
fit_ids, test_ids = train_test_split(film_ids, test_size=0.15, random_state=SEED, shuffle=True)
fit_rows = training_table.loc[training_table.TITLE_ID.isin(fit_ids)].copy()
test_rows = training_table.loc[training_table.TITLE_ID.isin(test_ids)].copy()

### 7 · Train with fitted preprocessing

The estimator contains a categorical encoder and numeric imputer **fitted on the training data**. These same fitted transformations must be used for inference; do not fit a new encoder or imputer on the separate inference records. A logarithmic target transformation handles skewed stream counts; predictions are returned in the original unit.

In [8]:
def make_estimator():
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    numeric = SimpleImputer(strategy="median")
    preprocessor = ColumnTransformer([
        ("categorical", categorical, CATEGORICAL_FEATURES),
        ("numeric", numeric, NUMERIC_FEATURES),
    ])
    forest = RandomForestRegressor(
        n_estimators=200, min_samples_leaf=4, random_state=SEED, n_jobs=-1
    )
    pipeline = Pipeline([("preprocessing", preprocessor), ("random_forest", forest)])
    return TransformedTargetRegressor(
        regressor=pipeline, func=np.log1p, inverse_func=np.expm1
    )

model_for_evaluation = make_estimator()
model_for_evaluation.fit(fit_rows[FEATURE_COLUMNS], fit_rows[TARGET])
pred_test = model_for_evaluation.predict(test_rows[FEATURE_COLUMNS])
naive_test = test_rows["may_streams"].to_numpy()
y_test = test_rows[TARGET].to_numpy()

### 8 · Compare with a simple baseline

For context, compare predictions for unseen films against the baseline that June streams equal May streams. The challenge focuses on using the supplied model correctly, **not** on improving its predictive accuracy.

In [9]:
def metric_summary(y_true, y_pred):
    return {
        "MAE": round(float(mean_absolute_error(y_true, y_pred)), 3),
        "RMSE": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 3),
        "WAPE": round(float(np.abs(y_true - y_pred).sum() / max(y_true.sum(), 1)), 4),
        "R2": round(float(r2_score(y_true, y_pred)), 4),
    }

metrics = pd.DataFrame({
    "May-as-June baseline": metric_summary(y_test, naive_test),
    "Random Forest": metric_summary(y_test, pred_test),
}).T
display(metrics)

,MAE,RMSE,WAPE,R2
May-as-June baseline,58.616,130.686,0.8490,-0.4772
Random Forest,42.996,93.818,0.6227,0.2387


### 9 · Fit the reference estimator on all development data

After evaluating on unseen films, the released model was trained on the entire development dataset and serialized as `artifacts/movie_consumption_model.pkl`. It contains the fitted preprocessing and the Random Forest, **but not** the preceding raw-data aggregation or join.

The following cell illustrates the final fit. This reference notebook does not write or replace the supplied artifact.

In [10]:
final_model = make_estimator()
final_model.fit(training_table[FEATURE_COLUMNS], training_table[TARGET])

# The supplied pickle was serialized from a fitted model of this form.
# This reference notebook intentionally does not overwrite it.

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...m_state=42))])
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the tr

### Applying the reference to the challenge

The separate raw inference files are `data/inference_movies.csv` and `data/inference_consumption.csv`. Read them independently, reproduce **May aggregation and movie-attribute joins** as illustrated above, and load the supplied fitted model for predictions. The required output has one row per May film–country–platform combination, with `TITLE_ID`, `country`, `platform`, and `predicted_june_streams`.

There is **no June inference input**, and retraining is not part of your task. See `artifacts/feature_schema.json` for the feature names and ordering expected by the saved model.